In [2]:
import os
import vertexai
from vertexai.vision_models import MultiModalEmbeddingModel, Video
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import time
import pandas as pd
import numpy as np
import subprocess
import imageio_ffmpeg
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

# --- NEW IMPORTS FOR PROPER SPLITTING & METRICS ---
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()

SEED = 2023 

In [3]:
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

seed_everything(SEED)

MASTER_DATASET = "out/master_dataset.csv"
EMBEDDINGS_FILE = "out/gemini_multimodal_embeddings.pt"
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 1e-4
INPUT_DIM = 1408
NUM_CLASSES = 2
PLOT_DIR = "plots"

os.makedirs(PLOT_DIR, exist_ok=True)

# --- NEW: GROUP SPLIT FUNCTION ---
def group_split(df: pd.DataFrame, test_size: float = 0.20, seed: int = SEED):
    """
    Split so that ALL utterances from a given file_id stay in the same partition.
    Prevents the cross-conversation leakage bug.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    groups = df["file_id"].values
    train_idx, test_idx = next(gss.split(df, groups=groups))
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)


class GeminiMultimodalDataset(Dataset):
    def __init__(self, df, embeddings_dict):
        self.X = []
        self.y = []

        for _, row in df.iterrows():
            sample_id = row["sample_id"]
            if sample_id in embeddings_dict:
                emb = embeddings_dict[sample_id]
                if isinstance(emb, torch.Tensor):
                    emb = emb.detach().cpu().numpy()
                self.X.append(emb)
                self.y.append(row["label"])

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
class GeminiClassifierHead(nn.Module):
    def __init__(self, input_dim=INPUT_DIM):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        return self.network(x)


def run_gemini_training():
    print("Loading dataset and embeddings...")
    df = pd.read_csv(MASTER_DATASET)
    embeddings_dict = torch.load(EMBEDDINGS_FILE, map_location="cpu")

    # --- UPDATED: Using GroupShuffleSplit to prevent data leakage ---
    train_df, test_df = group_split(df, test_size=0.20, seed=SEED)
    
    print(f"Train: {len(train_df)} utterances from {train_df['file_id'].nunique()} unique videos")
    print(f"Test : {len(test_df)} utterances from {test_df['file_id'].nunique()} unique videos")

    train_dataset = GeminiMultimodalDataset(train_df, embeddings_dict)
    test_dataset = GeminiMultimodalDataset(test_df, embeddings_dict)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    model = GeminiClassifierHead().to(device)

    # Handle class imbalance
    class_counts = train_df["label"].value_counts().sort_index().values
    weights = 1.0 / class_counts
    weights = weights / weights.sum()
    class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)

    train_losses, val_losses, val_accuracies = [], [], []

    print("\nStarting training...")
    for epoch in range(EPOCHS):
        # Training
        model.train()
        running_train_loss = 0.0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

        epoch_train_loss = running_train_loss / max(1, len(train_loader))
        train_losses.append(epoch_train_loss)

        # Validation
        model.eval()
        running_val_loss = 0.0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)

                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                running_val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())

        epoch_val_loss = running_val_loss / max(1, len(test_loader))
        epoch_val_acc = accuracy_score(val_targets, val_preds)

        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch [{epoch + 1}/{EPOCHS}] | "
                f"Train Loss: {epoch_train_loss:.4f} | "
                f"Val Loss: {epoch_val_loss:.4f} | "
                f"Val Acc: {epoch_val_acc * 100:.2f}%"
            )

    # Final evaluation
    f1 = f1_score(val_targets, val_preds)
    print("\n" + "=" * 50)
    print("RESULTS: Early Fusion Model (V2 Grouped Split)")
    print("=" * 50)
    print(f"Final Accuracy: {val_accuracies[-1] * 100:.2f}%")
    print(f"Final F1 Score: {f1:.4f}")
    print("-" * 50)
    print("Detailed Classification Report:")
    print(classification_report(val_targets, val_preds, target_names=["Calm (0)", "Conflict (1)"]))

    # Learning curves
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, EPOCHS + 1), train_losses, label="Training Loss", linewidth=2)
    plt.plot(range(1, EPOCHS + 1), val_losses, label="Validation Loss", linewidth=2, linestyle="--")
    plt.title("Training & Validation Loss", fontsize=14)
    plt.xlabel("Epochs")
    plt.ylabel("Cross-Entropy Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "early_fusion__learning_curves.png"), dpi=300)
    print(f"Learning curves saved to '{PLOT_DIR}/early_fusion_learning_curves.png'")
    plt.close()

    # Confusion matrix
    cm = confusion_matrix(val_targets, val_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Predicted Calm", "Predicted Conflict"],
        yticklabels=["Actual Calm", "Actual Conflict"],
        annot_kws={"size": 14}
    )
    plt.title("Confusion Matrix: Early Fusion", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "early_fusion_confusion_matrix.png"), dpi=300)
    print(f"Confusion matrix saved to '{PLOT_DIR}/early_fusion_confusion_matrix.png'")
    plt.close()


if __name__ == "__main__":
    run_gemini_training()

Loading dataset and embeddings...


/tmp/SLURM_7926173/ipykernel_3586366/331154533.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embeddings_dict = torch.load(EMBEDDINGS_FILE, map_location="cpu")


Train: 6109 utterances from 427 unique videos
Test : 1750 utterances from 107 unique videos
Training on device: cuda

Starting training...
Epoch [1/15] | Train Loss: 0.5479 | Val Loss: 0.5370 | Val Acc: 73.39%
Epoch [5/15] | Train Loss: 0.3513 | Val Loss: 0.4725 | Val Acc: 78.61%
Epoch [10/15] | Train Loss: 0.3071 | Val Loss: 0.4703 | Val Acc: 78.78%
Epoch [15/15] | Train Loss: 0.2854 | Val Loss: 0.4683 | Val Acc: 79.47%

RESULTS: Early Fusion Model (V2 Grouped Split)
Final Accuracy: 79.47%
Final F1 Score: 0.7604
--------------------------------------------------
Detailed Classification Report:
              precision    recall  f1-score   support

    Calm (0)       0.77      0.87      0.82       935
Conflict (1)       0.83      0.70      0.76       809

    accuracy                           0.79      1744
   macro avg       0.80      0.79      0.79      1744
weighted avg       0.80      0.79      0.79      1744

Learning curves saved to 'plots/early_fusion_learning_curves.png'
Confu